# Momants CSV Sentiment Processor

This notebook processes a Momants CSV export with `tabularisai/multilingual-sentiment-analysis`. The project contains no sample data: enter only the path to your own CSV below.

The processor groups messages by `conversation_id` and orders them chronologically within each conversation using `created_at`. Only customer messages (`from_agent == False`) are classified.

## 1. Import the processor

The processing code is in `momants_sentiment.py`, so you can use the same code from this notebook and from the command line.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from momants_sentiment import load_momants_csv, select_customer_messages, process_csv

## 2. Choose the CSV and output directory

Replace the example path with the path to your Momants export. By default, the output does not contain original message text.

In [ ]:
CSV_PATH = PROJECT_ROOT / "attached_assets" / "your_momants_export.csv"
OUTPUT_DIRECTORY = PROJECT_ROOT / "results"

print(f"Input: {CSV_PATH}")
print(f"Output directory: {OUTPUT_DIRECTORY}")

## 3. Check the structure first

This step does not start the model. It checks whether the CSV can be loaded and shows counts only.

In [ ]:
dataframe = load_momants_csv(CSV_PATH)
customer_messages = select_customer_messages(dataframe)

print(f"Message rows: {len(dataframe)}")
print(f"Usable customer messages: {len(customer_messages)}")
print(f"Conversations: {customer_messages['conversation_id'].nunique()}")

## 4. Determine starting and ending sentiment

This step loads the TabularisAI model and writes one table with the processing start timestamp in its name, for example `sentiment_per_conversation_20260901_143522_123456.csv`. Each conversation receives the sentiment of its first and last usable customer message. For a single customer message, starting and ending sentiment are based on the same model result, classified only once.

In [ ]:
conversation_results = process_csv(
    csv_path=CSV_PATH,
    output_directory=OUTPUT_DIRECTORY,
    batch_size=32,
)

print(f"Output file: {conversation_results.attrs['output_path']}")
conversation_results.head(10)